# CV Matcher - LoRA Fine-Tuning

Bu notebook, CV parsing ve CV-JD matching için **LoRA** ile fine-tune eder.

**Model:** Qwen2.5-1.5B-Instruct (açık kaynak, lisans gerekmez)

> **Not:** Gemma-2B fine-tuning'i de aynı kodla çalışır. Gemma'nın HuggingFace erişimi onay beklediği için Qwen kullanıyoruz.
> Gemma onayı çıkınca `--base-model google/gemma-2b-it` olarak değiştirip çalıştırabilirsin.

## Kullanım
1. **Runtime → Change runtime type** → **T4 GPU** seç
2. **Ctrl+F9** ile tüm hücreleri çalıştır
3. Fine-tuning bittiğinde adapter'lar otomatik indirilir

## Adım 1: GPU ve Kütüphaneler

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!pip install -q transformers accelerate peft trl datasets bitsandbytes scipy sentencepiece

## Adım 2: Projeyi ve Dataset'i Yükle

In [ ]:
!git clone https://github.com/ruveydagundogan/cvmatcher.git
%cd cvmatcher

import json
with open("backend/finetune/data/cv_parse_dataset.json") as f:
    cv_data = json.load(f)
with open("backend/finetune/data/cv_jd_match_dataset.json") as f:
    match_data = json.load(f)
print(f"CV Parse: {len(cv_data)} ornek")
print(f"CV-JD Match: {len(match_data)} ornek")

## Adım 3: CV Parsing Modeli Fine-Tune

**Hiperparametreler:**
- Model: Qwen2.5-1.5B-Instruct (Gemma için `--base-model google/gemma-2b-it`)
- LoRA rank (r): 8
- LoRA alpha: 16
- Epoch: 5
- Batch size: 4
- 4-bit quantization: True

In [ ]:
import sys
sys.path.append("backend/finetune")

BASE = "Qwen/Qwen2.5-1.5B-Instruct"

!python backend/finetune/train_lora.py \
    --base-model $BASE \
    --data backend/finetune/data/cv_parse_dataset.json \
    --output-dir /content/cvmatcher-lora/cv-parser-v1 \
    --mode cv-parse \
    --epochs 5 \
    --batch-size 4 \
    --max-length 512 \
    --lr 2e-4 \
    --quantize

print("CV Parse modeli fine-tune edildi!")

## Adım 4: CV-JD Matching Modeli Fine-Tune

In [ ]:
!python backend/finetune/train_lora.py \
    --base-model $BASE \
    --data backend/finetune/data/cv_jd_match_dataset.json \
    --output-dir /content/cvmatcher-lora/cv-jd-matcher-v1 \
    --mode cv-jd-match \
    --epochs 5 \
    --batch-size 4 \
    --max-length 512 \
    --lr 2e-4 \
    --quantize

print("CV-JD Match modeli fine-tune edildi!")

## Adım 5: Base vs Fine-Tuned Karşılaştırması

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

test_cv = """Python backend developer with 4 years experience.
Skilled in Django, FastAPI, PostgreSQL, Redis, Celery, Docker.
Built REST APIs serving 50K requests per minute.
Bachelor's in Software Engineering."""

messages = [
    {"role": "user", "content": f"Parse the following CV text and extract structured information: skills, experience, education, and a brief summary.\n\n{test_cv}"}
]

print("=" * 60)
print("BASE MODEL TESTI")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(BASE)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = base_model.generate(**inputs, max_new_tokens=256, temperature=0.1)
base_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(base_response[-500:] if len(base_response) > 500 else base_response)

In [ ]:
print("=" * 60)
print("FINE-TUNED MODEL (LoRA) TESTI")
print("=" * 60)

finetuned = PeftModel.from_pretrained(
    base_model, "/content/cvmatcher-lora/cv-parser-v1"
)
finetuned.eval()

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = finetuned.generate(**inputs, max_new_tokens=256, temperature=0.1)
ft_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(ft_response[-500:] if len(ft_response) > 500 else ft_response)

## Adım 6: Adapter'ları İndir

In [ ]:
import zipfile, os

zip_path = "/content/cvmatcher-lora.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk("/content/cvmatcher-lora"):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, "/content/cvmatcher-lora")
            zf.write(file_path, arcname)

print(f"Zip: {zip_path}")
from google.colab import files
files.download(zip_path)

## Ollama'ya Yükleme Talimatı

```bash
# Mac'te:
unzip ~/Downloads/cvmatcher-lora.zip -d backend/finetune/adapters/
bash backend/finetune/setup_ollama.sh ~/Downloads/cvmatcher-lora.zip

# Backend'i fine-tuned model ile başlat:
export OLLAMA_MODEL=cv-parser
go run cmd/server/main.go
```